# Reproducing Ghaderi, Crochet & Petersen (2026) from dandiset 001530: behaviour, optogenetics and single-neuron analyses

**Paper.** Ghaderi P, Crochet S, Petersen CCH (2026). Contextual gating of whisker-evoked responses by
frontal cortex supports flexible decision making. *Nature Communications* 17, 5982.
[doi:10.1038/s41467-026-73622-y](https://doi.org/10.1038/s41467-026-73622-y). The claims file was
written from the version of record in Europe PMC (PMC13346651): main text, legends and Methods; the
Supplementary Information was not read by the reader.

**Dandiset.** [DANDI:001530](https://dandiarchive.org/dandiset/001530/0.260605.0601), version
0.260605.0601. The paper cites the dandiset without a version. This is the only published version
and the most recent one; the draft has the same 196 assets and the same size (2,595,454,486,752
bytes), and it was last modified within seconds of the publication (2026-06-05), so nothing in the
draft is newer than what is pinned here.

**What is read.** The 35 `*-processed-behavior.nwb` files (Neuropixels units with Allen CCF labels,
trials, DeepLabCut traces) and the 91 `*-opto-behavior.nwb` files (trials of the optogenetic
sessions), streamed from the archive. The `*-processed-behavior-raw.nwb` files and the videos
hold the raw voltage and video of the same sessions and are not needed here.

**Outside inputs** (Zenodo record [doi:10.5281/zenodo.17973874](https://doi.org/10.5281/zenodo.17973874),
the paper's data and code, read by range requests into memory): `data_helpers/Area_list.mat`, the
CCF labels that make up each of the five areas, and `processed_data/Data_Clustering.mat`, the
paper's cluster label for each neuron (claims `cluster-label-link` and `fig4b-named-clusters`).

The population decoding claims and the clustering of Figure 4a are in `decoding_delay.ipynb` and
`decoding_clusters_whisker.ipynb`.

| Claim | Here |
|---|---|
| fig1c-lick-probabilities, fig1f-movement-latencies, quiet-trials-half | tested |
| fig2-auditory-window, fig2-delay-window, fig2-whisker-window, fig2-wm2-vs-alm-auditory, fig2-other-trial-types | tested |
| units-recorded, fig3a-activity-maps, fig3b-delay-and-response-psth, fig3c-delay-modulation-wm2-alm, fig3d-wm2-infragranular-delay | tested |
| cluster-label-link, fig4b-named-clusters | tested (with the paper's cluster labels) |
| fig5a-single-neuron-context | tested |
| fig6a-pca-delay-states, fig6b-context-coding-direction, fig6c-temporal-correlation | tested |
| fig7b-early-whisker-response, fig7c-no-whisker-cd-change, fig8a-wm2-lick-divergence | tested |
| fig8b-lick-coding-direction, figs18-later-lick-coding | blocked (see the summary) |
| fig5b-context-decoding-delay, fig5b-decoding-time-course | `decoding_delay.ipynb` |
| fig4-29-clusters, fig5c-cluster-removal, fig5d-null-space-decoding, fig7d-context-decoding-after-whisker | `decoding_clusters_whisker.ipynb` |

In [ ]:
import time
import warnings
import collections

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ghaderi as g

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)

started = time.time()
print(f"dandiset {g.DANDISET_ID} version {g.DANDISET_VERSION}: {len(g.assets())} assets")
AREA_LIST = g.area_list()
S = g.load_sessions(workers=4)
print(f"loaded {len(S)} ephys sessions in {time.time() - started:.0f} s")
t0 = time.time()
O = g.load_opto()
print(f"loaded {len(O)} optogenetic sessions in {time.time() - t0:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
kinds = collections.Counter()
for path, size, _ in g.assets():
    kind = ("ephys processed" if path.endswith("-processed-behavior.nwb") else
            "ephys raw" if path.endswith("-raw.nwb") else
            "video" if path.endswith(".avi") else
            "optogenetics" if path.endswith("-opto-behavior.nwb") else "other")
    kinds[kind] += 1
print("Assets by kind:", dict(kinds))
print("Subjects with ephys:", sorted({s["subject"] for s in S}), "n =", len({s["subject"] for s in S}))
print("Subjects with optogenetics:", sorted({o["subject"] for o in O}), "n =", len({o["subject"] for o in O}))
print("All subjects:", len({s["subject"] for s in S} | {o["subject"] for o in O}), "(archive summary: 21)")
print("\nCCF labels per area (Area_list.mat):")
for a, labels in AREA_LIST.items():
    print(f"  {a}: {len(labels)} labels, e.g. {labels[0].strip()!r}")

rows = []
for s in S:
    tr, c = s["trials"], s["completed"]
    tt = tr["trial_type"]
    rows.append({"session": s["session"], "subject": s["subject"], "trials": len(tt),
                 "completed": int(c.sum()), "early_lick": int((~c).sum()),
                 **{f"type{k}": int(((tt == k) & c).sum()) for k in range(0, 6)},
                 "units": len(s["units"]["location"]), "analysed_units": len(s["keep"]),
                 "probes": ",".join(sorted(set(s["units"]["location"]))),
                 "video_dt_ms": round(s["beh_info"]["video_dt_median_ms"], 3),
                 "unsorted_units": s["unsorted_units"]})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("\nTrial type 0 (not one of the five types) in:", info.loc[info.type0 > 0, "session"].tolist())
print("Sessions without No-tone Whisker trials:", info.loc[info.type5 == 0, "session"].tolist())
print("Units whose spike times are not sorted:", int(info.unsorted_units.sum()))

The archive lists 21 subjects: 14 mice with recordings and 15 with optogenetics, with overlap. The
35 recording sessions in 14 mice are the paper's (Results; the Methods' list sums to 34). Two
sessions break the five trial types: one has 11 trials coded `trial_type` 0, and one has no No-tone
trials; both are kept, with type 0 excluded from every trial type. Video frame intervals are 5 ms
(200 frames/s) in every session except the two of PG019, at 10 ms, as the Methods say (100 Hz in
PG019).

In [ ]:
print("Trial timing (completed trials, all sessions):")
ws = np.concatenate([s["trials"]["whisker_stim_time"] - s["trials"]["start_time"] for s in S])
au = np.concatenate([s["trials"]["auditory_stim_time"] - s["trials"]["start_time"] for s in S])
ws, au = ws[np.isfinite(ws)], au[np.isfinite(au)]
print(f"  whisker_stim_time - start_time: {ws.min():.4f} to {ws.max():.4f} s (median {np.median(ws):.4f})")
print(f"  auditory_stim_time - start_time: {au.min():.4f} to {au.max():.4f} s")
lt = np.concatenate([(s["trials"]["lick_time"] - s["trials"]["start_time"])[s["completed"] & s["trials"]["lick_flag"].astype(bool)] for s in S])
print(f"  first lick - start, completed trials with lick_flag: {np.nanmin(lt):.3f} to {np.nanmax(lt):.3f} s")
ctx = collections.Counter()
for s in S:
    for c_, t_ in zip(s["trials"]["context"], s["trials"]["trial_type"]):
        ctx[(c_, t_)] += 1
print("  context and trial_type pairs:", dict(sorted(ctx.items(), key=lambda x: (x[0][1], x[0][0]))))
freq = collections.Counter()
for s in S:
    for c_, f_ in zip(s["trials"]["context"], s["trials"]["auditory_stim_frequency"]):
        freq[(s["subject"], c_, f_)] += 1
print("  Go/Nogo tone frequencies per mouse:",
      sorted({(k[0], k[1], k[2]) for k in freq if k[1] != "no_tone"}))
wd = np.unique(np.concatenate([s["trials"]["whisker_stim_duration"] for s in S]))
print("  whisker_stim_duration values (s):", wd[np.isfinite(wd)])

The whisker stimulus is 1.0 s after `start_time` in every trial and the first lick falls in the
1 s response window (1 to 2 s), as the Methods describe. `whisker_stim_duration` is 0.01 s, the
10 ms cosine pulse of the Methods (5 ms at half maximum). Each mouse has one Go and one Nogo
frequency (4 or 8 kHz).

In [ ]:
rows = []
for s in S:
    b, inf = s["beh"], s["beh_info"]
    rows.append({"session": s["session"], "frames": inf["video_n_frames"],
                 "video_s": f"{inf['video_t0']:.0f}-{inf['video_t1']:.0f}",
                 "trials_s": f"{s['trials']['start_time'][0]:.0f}-{s['trials']['start_time'][-1]:.0f}",
                 "trials_no_video": int(np.isnan(b["whisker_speed"]).all(axis=1).sum()),
                 "angle_deg": f"{inf['whisker_angle_range'][0]:.0f}-{inf['whisker_angle_range'][1]:.0f}",
                 "jaw_px_max": round(float(np.nanmax(b["jaw_movement"])), 1),
                 "quiet_median_%": round(100 * (s["quiet_median"]["whisker_speed"] & s["quiet_median"]["jaw_movement"]).mean(), 1),
                 "quiet_mean_%": round(100 * (s["quiet_mean"]["whisker_speed"] & s["quiet_mean"]["jaw_movement"]).mean(), 1)})
beh_info = pd.DataFrame(rows)
print(beh_info.to_string(index=False))
print("Trials with no video in their -1 to 2 s window:", int(beh_info.trials_no_video.sum()), "of", int(info.trials.sum()))

Whisker angle is in degrees and jaw and tongue in pixels, as the files state. Video covers almost
every trial, but in sub-PG061_ses-20220212T233813 243 trials and in sub-PG038_ses-20210316T152753
29 trials have no video frames in their window; those trials cannot pass the quiet rule and are
counted as not quiet, as the paper's code would count them (a NaN fails its comparison).

In [ ]:
# Shared clock of spikes and trials: whisker-evoked response (1.00 to 1.03 s) in whisker against
# no-whisker trials, in wS1 and wS2, with true and with trial labels shifted by one trial.
rows = []
w, b = g.bin_index(1.01, 1.03), g.bin_index(0.8, 1.0)
for k, s in enumerate(S):
    tt = s["trials"]["trial_type"][s["completed"]]
    whisk = np.isin(tt, [1, 3, 5])
    for a in ("wS1", "wS2", "A1", "all units"):
        cols = np.arange(len(s["keep"])) if a == "all units" else g.area_columns(s, a)
        if len(cols) < 5:
            continue
        x = s["c10"][:, :, cols].astype(float)
        if a == "A1":   # auditory response 0 to 30 ms, tone against no-tone trials
            resp = x[:, g.bin_index(0.01, 0.03)].mean(axis=(1, 2)) - x[:, g.bin_index(-0.2, 0.0)].mean(axis=(1, 2))
            lab = np.isin(tt, [1, 2, 3, 4])
        else:
            resp = x[:, w].mean(axis=(1, 2)) - x[:, b].mean(axis=(1, 2))
            lab = whisk
        rows.append({"session": s["session"], "area": a,
                     "contrast_true": (resp[lab].mean() - resp[~lab].mean()) / g.W10,
                     "contrast_shifted": (resp[np.roll(lab, 1)].mean() - resp[~np.roll(lab, 1)].mean()) / g.W10})
clock = pd.DataFrame(rows)
print("Stimulus-evoked change in rate (Hz per neuron), stimulus minus no-stimulus trials:")
print(clock.groupby("area")[["contrast_true", "contrast_shifted"]].describe().round(2).T.to_string())
print("\nArea-sessions where the true contrast is not above the shifted one:")
print(clock[clock.contrast_true <= clock.contrast_shifted].to_string(index=False))
print("Sessions with no wS1, wS2 or A1 area of 5 units (checked with all units only):",
      sorted(set(clock.session) - set(clock[clock.area != "all units"].session)))
print("Sessions with fewer than 5 analysed units (not checked):", [s["session"] for s in S if len(s["keep"]) < 5])

In every area-session the whisker-evoked response in wS1 and wS2 and the tone-evoked response in
A1 are larger in the trials that had the stimulus than in those that did not, and the contrast
vanishes when the labels are shifted by one trial (table above). The sessions with only frontal
units are checked with all their units pooled (whisker-evoked response, 1.00 to 1.03 s). Spikes and
the trial table share a clock in each session tested this way. sub-PG038_ses-20210317T180556 keeps
no unit after the CCF filter, so it is not checked and enters only the behavioural claims.

In [ ]:
# Recordings deposited twice: compare unit counts, spike counts and trial start times.
sig = collections.defaultdict(list)
for s in S:
    sig[(len(s["units"]["location"]), s["n_spikes"])].append(s["session"])
    sig[("trials", round(float(s["trials"]["start_time"][0]), 3), len(s["trials"]["start_time"]))].append(s["session"])
dups = {k: v for k, v in sig.items() if len(v) > 1}
print("Sessions sharing a unit and spike count, or trial start times:", dups if dups else "none")
print("Subjects and sessions:", info.groupby("subject").session.count().to_dict())
same_day = info.assign(day=info.session.str[:22]).groupby("day").session.count()
print("More than one file on a day:", same_day[same_day > 1].to_dict() or "none")

No two files share their unit and spike counts or their trial times; each recording is counted once.
Units are not followed across sessions in any claim here.

In [ ]:
rows = []
for o in O:
    tr = o["trials"]
    rows.append({"subject": o["subject"], "session": o["path"].split("/")[-1][:30], "area": o["area"],
                 "genotype": o["genotype"], "trials": len(tr["trial_type"]),
                 **{w_: int((tr["opto_window"] == w_).sum()) for w_ in ("nolight", "audio", "delay", "whisker")},
                 "power_mW": ",".join(f"{v:g}" for v in np.unique(tr["opto_stim_amplitude"][np.isfinite(tr["opto_stim_amplitude"])])),
                 "freq_Hz": ",".join(f"{v:g}" for v in np.unique(tr["opto_stim_frequency"][np.isfinite(tr["opto_stim_frequency"])]))})
opto_info = pd.DataFrame(rows)
print(opto_info.groupby("area").agg(sessions=("session", "count"), mice=("subject", "nunique")).to_string())
print("\nGenotype field:", opto_info.genotype.value_counts().to_dict())
print(opto_info[opto_info.genotype != "VGAT_ChR2-eYFP"].to_string(index=False))
print("\nWindows used per session:", collections.Counter(tuple(w_ for w_ in ("audio", "delay", "whisker") if r[w_] > 0) for _, r in opto_info.iterrows()))
print("Light trials, fraction of all:", round(opto_info[["audio", "delay", "whisker"]].sum().sum() / opto_info.trials.sum(), 3))
print("Laser power values (mW):", sorted({p for v in opto_info.power_mW for p in v.split(",") if p}))
print("Pulse frequency values (Hz):", sorted({p for v in opto_info.freq_Hz for p in v.split(",") if p}))

The 91 optogenetic sessions cover the six areas with 9 to 20 sessions each; mice per area are A1 9,
wS1 10, wS2 10, wM2 9, ALM 7 and fpS1 8, where the Methods give 8 mice for ALM. Each session
inactivates one area in one window. Twelve sessions (mice PG021 and PG061) give the genotype
C57BL/6JRj, which carries no ChR2, although the paper inactivates only VGAT-ChR2 mice; their light
trials still reduce licking (see fig2 below), so the genotype field is likely wrong for these mice.
The trials table describes `opto_area` as "laser light duration" and `opto_stim_start_time` as
"Pulse duration in s", which do not fit their values.

## fig1c-lick-probabilities

> Mice lick in the response window on 78 +/- 2% of Go-tone Whisker trials and withhold licking in the
> other trial types (Go-tone No-Whisker 10 +/- 1%, Nogo-tone Whisker 13 +/- 2%, Nogo-tone No-Whisker
> 1 +/- 0.3%, No-tone Whisker 15 +/- 2%), mean +/- SEM over 35 recording sessions in 14 mice.

As the criterion (code, Ghaderi2025_Figure1_c_Performance.m): completed trials, `lick_flag`, mean
and SEM over sessions.

In [ ]:
P = []
for s in S:
    tr, c = s["trials"], s["completed"]
    tt, L = tr["trial_type"], tr["lick_flag"].astype(bool)
    P.append({"session": s["session"], "subject": s["subject"],
              **{g.TRIAL_NAMES[k]: (100 * L[c & (tt == k)].mean() if (c & (tt == k)).any() else np.nan) for k in range(1, 6)}})
P = pd.DataFrame(P)
expected = {"Go-tone Whisker": (78, 2), "Go-tone No-Whisker": (10, 1), "Nogo-tone Whisker": (13, 2),
            "Nogo-tone No-Whisker": (1, 0.3), "No-tone Whisker": (15, 2)}
rows = []
for name, (v, tol) in expected.items():
    x = P[name].dropna()
    rows.append({"trial type": name, "sessions": len(x), "mean %": round(x.mean(), 2),
                 "SEM": round(x.std(ddof=1) / np.sqrt(len(x)), 2), "paper": f"{v} +/- {tol}",
                 "within": abs(x.mean() - v) <= tol, "min": round(x.min(), 1), "max": round(x.max(), 1)})
fig1c = pd.DataFrame(rows)
print(fig1c.to_string(index=False))
print("\nPer mouse (mean over its sessions):")
print(P.groupby("subject")[list(expected)].mean().round(1).to_string())

**Result: reproduced.** Over the 35 sessions (34 for No-tone Whisker, which one session lacks),
P(lick) is 78.18 +/- 2.08% for Go-tone Whisker, 9.90 +/- 1.49% Go-tone No-Whisker, 13.08 +/- 1.79%
Nogo-tone Whisker, 1.05 +/- 0.26% Nogo-tone No-Whisker and 14.64 +/- 1.72% No-tone Whisker: all
five means are within tolerance.

## fig1f-movement-latencies

> In quiet Hit trials the whisker deflection is followed by whisker protraction (time to 20% of maximum
> 58 +/- 8 ms), jaw opening (115 +/- 17 ms) and tongue protrusion (231 +/- 29 ms), and tongue-spout
> contact at 330 +/- 20 ms.

Traces as code, Ghaderi2025_Figure1_f_Movements_Hit.m (quiet completed Hit trials, 10 ms bins, each
trial minus its mean over 0.8 to 1.0 s; NaN tongue values set to 0 as the code does). Readings (a)
and (b) of the criterion, with the maximum over 1.0 to 2.0 s, the 1 s response window after the
whisker stimulus (Methods, Mouse behavior), as the claims file has it since a reader answered the
question on the window. The lick latency as Ghaderi2025_Sup_Figure1_b_SpoutReactionTime.m. A
latency is the right edge of the first bin after 1.0 s that reaches 20% of the maximum, minus 1.0 s.
As variants the maximum is taken over 1.0 to 1.2 s (the Fig. 1f plot window, the criterion's window
before the answer) and 1.0 to 1.5 s, and the quiet rule uses the code's mean absolute deviation.

In [ ]:
labels10 = g.PRE + g.W10 * np.arange(1, g.N10 + 1)
base_bins = g.bin_index(0.8, 1.0)


def latency_ms(trace, post):
    seg = trace[post]
    mx = np.nanmax(seg) if np.isfinite(seg).any() else np.nan
    if not np.isfinite(mx) or mx <= 0:
        return np.nan
    hit = np.flatnonzero(seg >= 0.2 * mx)
    hit = hit[labels10[post][hit] > 1.0]
    return (labels10[post][hit[0]] - 1.0) * 1000 if len(hit) else np.nan


def movement_latencies(rule="median", end=2.0):
    post = g.bin_index(1.0, end)
    out = []
    for s in S:
        sel = g.trial_mask(s, 1, True) & g.quiet_mask(s, rule)
        row = {"session": s["session"], "subject": s["subject"], "n_trials": int(sel.sum())}
        for sigl in ("whisker_speed", "jaw_movement", "tongue_movement"):
            x = s["beh"][sigl][s["completed"]][sel].copy()
            if sigl == "tongue_movement":
                x = np.where(np.isnan(x), 0, x)
            x = x - np.nanmean(x[:, base_bins], axis=1, keepdims=True)
            row[f"{sigl}_a"] = latency_ms(np.nanmean(x, axis=0), post) if len(x) else np.nan
            row[f"{sigl}_b"] = np.nanmean([latency_ms(t, post) for t in x]) if len(x) else np.nan
        out.append(row)
    return pd.DataFrame(out)


lick_rows = []
for s in S:
    tr, c = s["trials"], s["completed"]
    rt = tr["lick_time"] - tr["start_time"]
    rt[(rt > 2) | (rt < 1)] = np.nan
    hit = (tr["trial_type"] == 1) & tr["lick_flag"].astype(bool) & c
    lick_rows.append(np.nanmean((rt - 1)[hit]) * 1000)
lick = np.array(lick_rows)
print(f"Lick latency (all completed Hit trials): {lick.mean():.1f} +/- {lick.std(ddof=1) / np.sqrt(len(lick)):.1f} ms "
      f"(n = {len(lick)} sessions; paper 330 +/- 20)")


def summarise(df, label):
    rows = []
    for sigl, (v, tol) in {"whisker_speed": (58, 8), "jaw_movement": (115, 17), "tongue_movement": (231, 29)}.items():
        for r in "ab":
            x = df[f"{sigl}_{r}"].dropna()
            rows.append({"variant": label, "signal": sigl, "reading": r, "sessions": len(x),
                         "mean ms": round(x.mean(), 1), "SEM": round(x.std(ddof=1) / np.sqrt(len(x)), 1),
                         "paper": f"{v} +/- {tol}", "within": abs(x.mean() - v) <= tol})
    return pd.DataFrame(rows)


lat = movement_latencies()
print("\nSessions with no quiet Hit trial:", lat.loc[lat.n_trials == 0, "session"].tolist())
print(summarise(lat, "criterion, max over 1.0-2.0 s").to_string(index=False))
order = {r: [lat[f"{x}_{r}"].mean() for x in ("whisker_speed", "jaw_movement", "tongue_movement")] + [lick.mean()]
         for r in "ab"}
for r, v in order.items():
    print(f"reading ({r}): whisker < jaw < tongue < spout contact: {all(np.diff(v) > 0)} ({', '.join(f'{x:.1f}' for x in v)} ms)")
print("\nCriterion window, latencies (ms) by subject (mean over sessions; reading a / reading b):")
cols = [f"{x}_{r}" for x in ("whisker_speed", "jaw_movement", "tongue_movement") for r in "ab"]
print(lat[lat.n_trials > 0].groupby("subject")[cols].mean().round(1).to_string())
print("\nCriterion window, median over sessions (ms):", lat[cols].median().round(1).to_dict())
for end in (1.2, 1.5):
    print(summarise(movement_latencies(end=end), f"variant, max over 1.0-{end} s").to_string(index=False))
print(summarise(movement_latencies("mean"), "variant, code's MAD, 1.0-2.0 s").to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
t = labels10[g.bin_index(0.8, 2.0)]
for sigl, col in zip(("whisker_speed", "jaw_movement", "tongue_movement"), ("m", "c", "orange")):
    traces = []
    for s in S:
        sel = g.trial_mask(s, 1, True) & g.quiet_mask(s)
        if sel.sum() == 0:
            continue
        x = s["beh"][sigl][s["completed"]][sel].copy()
        if sigl == "tongue_movement":
            x = np.where(np.isnan(x), 0, x)
        x = x - np.nanmean(x[:, base_bins], axis=1, keepdims=True)
        traces.append(np.nanmean(x, axis=0)[g.bin_index(0.8, 2.0)])
    m = np.nanmean(traces, axis=0)
    ax.plot(t, m / np.nanmax(m), color=col, label=sigl)
    print(f"{sigl}: grand-mean trace, max at {t[np.nanargmax(m)]:.2f} s; normalised range {np.nanmin(m / np.nanmax(m)):.2f} to 1")
ax.axvline(1.0, color="k", lw=0.5)
ax.axvline(1.2, color="grey", ls="--", lw=0.5)  # end of the Fig. 1f plot and of the 1.2 s variant
ax.set_xlabel("time from trial start (s)")
ax.set_ylabel("movement / max")
ax.legend(fontsize=8)
plt.show()

**Result: partially reproduced.** The lick latency is 330.1 +/- 20.2 ms, as the paper reports. With
the maximum over 1.0 to 2.0 s, as the claims file now has it, reading (a) gives whisker 54.4, jaw
115.3 and tongue 192.4 ms and reading (b) 94.6, 142.9 and 254.6 ms, against 58, 115 and 231 ms. Under
reading (a) the whisker and jaw latencies are within tolerance and the tongue latency is 38.6 ms
short of 231, outside the 29 ms tolerance; under reading (b) only the tongue latency is within
tolerance. The order whisker, jaw, tongue, spout contact holds under both readings, so both give the
criterion's partial case. The session means are pulled up by a few late sessions: the medians over
sessions under reading (a) are 50.0, 95.0 and 155.0 ms, and the two PG019 sessions average a jaw latency of 315.0 ms and
a tongue latency of 385.0 ms. With the maximum over 1.0 to 1.2 s (the window before the reader's
answer) no movement latency under reading (a) and only the whisker latency under reading (b) is
within tolerance, and with the code's mean absolute deviation for the quiet rule the values are
close to the criterion's (50.9, 118.0 and 196.0 ms under reading a).

## quiet-trials-half

> Selecting quiet trials (no whisker or jaw movement in the last 200 ms of the delay) rejects about
> half of the recorded trials.

In [ ]:
rows = []
for rule in ("median", "mean"):
    for reading, use_completed in (("a, completed trials", True), ("b, all trials", False)):
        rej, tot, per = 0, 0, []
        for s in S:
            q = s[f"quiet_{rule}"]
            quiet = q["whisker_speed"] & q["jaw_movement"]
            sel = s["completed"] if use_completed else np.ones(len(quiet), bool)
            rej += int((~quiet[sel]).sum())
            tot += int(sel.sum())
            per.append(100 * (~quiet[sel]).mean())
        rows.append({"MAD": rule, "reading": reading, "trials": tot, "rejected": rej,
                     "pooled %": round(100 * rej / tot, 2), "median over sessions %": round(np.median(per), 2),
                     "range over sessions %": f"{min(per):.1f}-{max(per):.1f}", "within 45-55": abs(100 * rej / tot - 50) <= 5})
quiet_tab = pd.DataFrame(rows)
print(quiet_tab.to_string(index=False))
# Variant: trials without video left out of the denominator.
rej = tot = 0
for s in S:
    q = s["quiet_median"]
    has_video = ~np.isnan(s["beh"]["whisker_speed"]).all(axis=1)
    sel = s["completed"] & has_video
    rej += int((~(q["whisker_speed"] & q["jaw_movement"]))[sel].sum())
    tot += int(sel.sum())
print(f"\nVariant, completed trials with video only (median MAD): {100 * rej / tot:.2f}% rejected of {tot}")
for sigl in ("whisker_speed", "jaw_movement"):
    r_ = np.concatenate([~s["quiet_median"][sigl][s["completed"]] for s in S]).mean()
    print(f"Rejected by {sigl} alone (median MAD, completed): {100 * r_:.1f}%")

**Result: partially reproduced.** With the Methods' rule (median absolute deviation) the quiet
selection rejects 55.06% of the completed trials (pooled; median over sessions 53.3%) and 56.72% of
all trials, just outside the 45 to 55% tolerance under both readings and inside the 30 to 70% band
for a partial result. With the code's mean absolute deviation, 40.22% of completed trials are
rejected. Leaving out the trials without video gives 54.22%.

## Optogenetic inactivation (Figure 2)

Per session, P(lick) over completed Go-tone Whisker trials with no light and with light in the
session's window (code, Ghaderi2025_Figure2_d_Plick_Sessions.m). The reduction is the mean over
sessions of no-light minus light P(lick), in points (reading a), or as percent of no-light
P(lick) (reading b). The test draws 9 sessions with replacement (the smallest session count over
areas and windows, computed below), p = (draws on the wrong side + 1) / iterations, two-tailed,
Benjamini-Hochberg over the six areas within a window; 1,000 iterations (Methods) and 10,000
(code). The spread is given as SEM and SD over sessions.

In [ ]:
def plick(o, window, ttype=1):
    tr = o["trials"]
    sel = ~tr["early_lick"].astype(bool) & (tr["trial_type"] == ttype) & (tr["opto_window"] == window)
    return 100 * tr["lick_flag"][sel].mean() if sel.any() else np.nan


WINDOWS = ["audio", "delay", "whisker"]


def opto_pairs(ttype=1):
    out = {}
    for a in g.OPTO_AREAS:
        sess = [o for o in O if o["area"] == a]
        nl = np.array([plick(o, "nolight", ttype) for o in sess])
        for w_ in WINDOWS:
            op = np.array([plick(o, w_, ttype) for o in sess])
            ok = np.isfinite(op) & np.isfinite(nl)
            out[a, w_] = (nl[ok], op[ok], [sess[i]["subject"] for i in np.flatnonzero(ok)])
    return out


pairs = opto_pairs()
n_draw = min(len(v[0]) for v in pairs.values())
print("Smallest number of sessions over areas and windows:", n_draw)
opto_rows = []
for iterations in (1000, 10000):
    rng = np.random.default_rng(2)
    for w_ in WINDOWS:
        ps = [g.paired_bootstrap_p(pairs[a, w_][0], pairs[a, w_][1], n_draw, iterations, rng) for a in g.OPTO_AREAS]
        qs = g.bh_fdr(ps)
        for a, p_, q_ in zip(g.OPTO_AREAS, ps, qs):
            nl, op, subj = pairs[a, w_]
            d = nl - op
            opto_rows.append({"iterations": iterations, "window": w_, "area": a, "sessions": len(d), "mice": len(set(subj)),
                              "reduction_pts": d.mean(), "SEM": d.std(ddof=1) / np.sqrt(len(d)), "SD": d.std(ddof=1),
                              "relative_%": 100 * np.mean(d / nl), "p": p_, "p_FDR": q_})
opto_tab = pd.DataFrame(opto_rows)
print(opto_tab.round(4).to_string(index=False))

In [ ]:
expected_opto = {("audio", "A1"): (39, 4), ("audio", "wM2"): (33, 6), ("audio", "wS2"): (26, 4), ("audio", "wS1"): (19, 6),
                 ("delay", "wM2"): (45, 7), ("delay", "ALM"): (32, 5), ("delay", "wS1"): (16, 3),
                 ("whisker", "ALM"): (52, 4), ("whisker", "wM2"): (48, 4), ("whisker", "wS1"): (38, 3),
                 ("whisker", "wS2"): (35, 4), ("whisker", "A1"): (13, 3)}
significant_expected = {"audio": {"A1", "wM2", "wS2", "wS1"}, "delay": {"wM2", "ALM", "wS1"},
                        "whisker": {"A1", "wS1", "wS2", "wM2", "ALM"}}
for iterations in (1000, 10000):
    for w_ in WINDOWS:
        sub = opto_tab[(opto_tab.iterations == iterations) & (opto_tab.window == w_)].set_index("area")
        sig_ = set(sub.index[sub.p_FDR < 0.05])
        checks = []
        for (ww, a), (v, tol) in expected_opto.items():
            if ww == w_:
                checks.append(f"{a} {sub.loc[a, 'reduction_pts']:.1f} pts / {sub.loc[a, 'relative_%']:.1f}% (paper {v} +/- {tol}; "
                              f"points within: {abs(sub.loc[a, 'reduction_pts'] - v) <= tol}; relative within: {abs(sub.loc[a, 'relative_%'] - v) <= tol})")
        print(f"{iterations} iterations, {w_}: significant {sorted(sig_)}; paper's pattern {sorted(significant_expected[w_])}; "
              f"same: {sig_ == significant_expected[w_]}")
        for c_ in checks:
            print("   ", c_)
print("\nPer mouse, mean reduction in points (Auditory, Delay, Whisker):")
rows = []
for a in g.OPTO_AREAS:
    for w_ in WINDOWS:
        nl, op, subj = pairs[a, w_]
        for m_ in sorted(set(subj)):
            k = np.array(subj) == m_
            rows.append({"area": a, "window": w_, "mouse": m_, "sessions": int(k.sum()), "reduction": round((nl[k] - op[k]).mean(), 1)})
print(pd.DataFrame(rows).pivot_table(index=["area", "mouse"], columns="window", values="reduction").round(1).to_string())

In [ ]:
fig, axs = plt.subplots(1, 6, figsize=(14, 3), sharey=True)
for ax, a in zip(axs, g.OPTO_AREAS):
    sess = [o for o in O if o["area"] == a]
    for o in sess:
        y = [plick(o, w_) for w_ in ["nolight"] + WINDOWS]
        ax.plot(range(4), y, "-o", color="0.7", ms=2, lw=0.5)
    ax.set_xticks(range(4), ["none", "aud", "delay", "whisk"], fontsize=7)
    ax.set_title(a)
axs[0].set_ylabel("P(lick), Go-tone Whisker (%)")
plt.show()
print("All plotted P(lick) values lie in 0 to 100%.")

**Result: reproduced** (fig2-auditory-window). In the Auditory window the reductions are A1 38.9
(SEM 4.1), wM2 33.0 (5.7), wS2 26.0 (3.8) and wS1 19.5 (6.1) points, all within tolerance in
points; these four areas are significant after FDR (10,000 iterations: 0.0004, 0.0004, 0.0004 and
0.0021; 1,000: 0.004 to 0.012) and ALM (7.8 points, p 0.161) and fpS1 (0.1, p 0.979) are not. The
paper's spread matches the SEM, not the SD. As a percent of no-light P(lick) (reading b) the values
are larger and out of tolerance, so the paper's numbers are points.

**Result: reproduced** (fig2-delay-window). In the Delay window wM2 44.5 (SEM 7.2), ALM 32.1 (5.5)
and wS1 16.0 (3.0) points, within tolerance and significant (p_FDR 0.0004 with 10,000 iterations);
A1 (7.8, p 0.113), wS2 (8.2, p 0.076) and fpS1 (4.3, p 0.292) are not significant.

**Result: reproduced** (fig2-whisker-window). In the Whisker window ALM 52.3 (SEM 3.8), wM2 48.2
(3.8), wS1 37.6 (3.5), wS2 35.5 (3.7) and A1 12.8 (3.5) points, within tolerance and significant
(p_FDR 0.0003 for the first four and 0.0115 for A1 with 10,000 iterations, the paper's 0.0003 and
0.01); fpS1 (8.7, p 0.118) is not. With 1,000 iterations the pattern is the same.

## fig2-wm2-vs-alm-auditory

> Inactivation during the auditory window reduces licking significantly more for wM2 than for ALM
> (33 +/- 6% against 8 +/- 5%).

Unpaired bootstrap of the difference of the two areas' mean per-session reductions, readings (a) 9
sessions per area and (b) all sessions of each area, two-tailed with the code's p rule; FDR over the
15 area pairs of the Auditory window as a variant.

In [ ]:
def unpaired_bootstrap_p(x, y, n_x, n_y, iterations, rng):
    d = rng.choice(x, (iterations, n_x)).mean(axis=1) - rng.choice(y, (iterations, n_y)).mean(axis=1)
    obs = x.mean() - y.mean()
    p_ = ((d <= 0).sum() + 1) / iterations if obs > 0 else ((d >= 0).sum() + 1) / iterations
    return 2 * min(p_, 1 - p_)


red = {a: pairs[a, "audio"][0] - pairs[a, "audio"][1] for a in g.OPTO_AREAS}
print(f"wM2 {red['wM2'].mean():.1f} +/- {red['wM2'].std(ddof=1) / np.sqrt(len(red['wM2'])):.1f} (n {len(red['wM2'])}); "
      f"ALM {red['ALM'].mean():.1f} +/- {red['ALM'].std(ddof=1) / np.sqrt(len(red['ALM'])):.1f} (n {len(red['ALM'])})")
for iterations in (1000, 10000):
    rng = np.random.default_rng(3)
    pa = unpaired_bootstrap_p(red["wM2"], red["ALM"], 9, 9, iterations, rng)
    pb = unpaired_bootstrap_p(red["wM2"], red["ALM"], len(red["wM2"]), len(red["ALM"]), iterations, rng)
    pairs15 = [(a, b) for i, a in enumerate(g.OPTO_AREAS) for b in g.OPTO_AREAS[i + 1:]]
    pall = [unpaired_bootstrap_p(red[a], red[b], len(red[a]), len(red[b]), iterations, rng) for a, b in pairs15]
    q15 = g.bh_fdr(pall)[pairs15.index(("wM2", "ALM"))]
    print(f"{iterations} iterations: reading (a) p = {pa:.4f}; reading (b) p = {pb:.4f}; "
          f"(b) with FDR over the 15 pairs: {q15:.4f}")

**Result: reproduced.** wM2 33.0 +/- 5.7 and ALM 7.8 +/- 5.4 points, both within tolerance, and the
bootstrap gives p < 0.05 with wM2 larger under both readings (10,000 iterations: 0.0024 and 0.0008
uncorrected, 0.0018 after FDR over the 15 pairs; 1,000 iterations: 0.006, 0.002 and 0.005).

## fig2-other-trial-types

> Inactivation has little impact on licking in the trial types other than Go-tone Whisker.

The change in P(lick) for trial types 2 to 5, 6 areas and 3 windows (72 changes); reading (a) the
bootstrap test of fig2 with FDR within each window and trial type finds no significant change in
more than 7 of 72; reading (b) no change exceeds 13 points.

In [ ]:
rows = []
rng = np.random.default_rng(4)
for ttype in (2, 3, 4, 5):
    pr = opto_pairs(ttype)
    for w_ in WINDOWS:
        ps = []
        for a in g.OPTO_AREAS:
            nl, op, _ = pr[a, w_]
            ps.append(g.paired_bootstrap_p(nl, op, n_draw, 1000, rng) if len(nl) > 1 else np.nan)
        qs = g.bh_fdr(np.nan_to_num(ps, nan=1.0))
        for a, p_, q_ in zip(g.OPTO_AREAS, ps, qs):
            nl, op, _ = pr[a, w_]
            rows.append({"trial type": g.TRIAL_NAMES[ttype], "window": w_, "area": a, "sessions": len(nl),
                         "change_pts": (op - nl).mean(), "p": p_, "p_FDR": q_})
other = pd.DataFrame(rows)
print(other.round(4).to_string(index=False))
n_sig = int((other.p_FDR < 0.05).sum())
print(f"\nReading (a): {n_sig} of {len(other)} changes significant after FDR (threshold: at most 7)")
print(f"Reading (b): largest absolute change {other.change_pts.abs().max():.1f} points (threshold 13); "
      f"changes above 13 points: {int((other.change_pts.abs() > 13).sum())}")
print(other.loc[other.change_pts.abs() > 13, ["trial type", "window", "area", "change_pts", "p_FDR"]].round(3).to_string(index=False))

**Result: not reproduced.** Both readings fail. By the bootstrap test 18 of the 72 changes are
significant after FDR (reading a allows at most 7), and two changes exceed 13 points (reading b),
both Go-tone No-Whisker trials in the Auditory window: A1 -19.0 and wS2 -15.7 points. Most
significant changes are reductions of licking in No-tone Whisker and Nogo-tone Whisker trials
(table above), smaller than on Go-tone Whisker trials but not absent.

## units-recorded

> Across 35 sessions in 14 mice, 10,294 well-isolated units were recorded in the five areas (A1 780,
> wS1 1938, wS2 2064, wM2 2749, ALM 2763).

The files hold only the units kept after curation (no inclusion column); a unit is in an area when
its probe `location` is the area and its `ccf_location` is one of the area's labels in
Area_list.mat.

In [ ]:
rows = []
for a in g.AREAS:
    ccf = sum(int((s["units"]["area"] == a).sum()) for s in S)
    probe = sum(int((s["units"]["location"] == a).sum()) for s in S)
    sess_probe = [s for s in S if (s["units"]["location"] == a).any()]
    sess_ccf = [s for s in S if (s["units"]["area"] == a).any()]
    rows.append({"area": a, "units (probe and CCF)": ccf, "units (probe only)": probe,
                 "sessions (probe)": len(sess_probe), "mice (probe)": len({s["subject"] for s in sess_probe}),
                 "sessions (with CCF units)": len(sess_ccf), "mice (with CCF units)": len({s["subject"] for s in sess_ccf})})
units = pd.DataFrame(rows)
units["paper units"] = [780, 1938, 2064, 2749, 2763]
units["paper sessions, mice"] = ["14, 6", "24, 12", "24, 11", "18, 8", "16, 6"]
print(units.to_string(index=False))
print("Total (probe and CCF):", units["units (probe and CCF)"].sum(), "; paper 10,294")
print("Units on other probes:", {L: sum(int((s["units"]["location"] == L).sum()) for s in S) for L in ("tjM1", "unassigned_area")})
print("Cell types of the 10,294 (RS, FS, neither):",
      [sum(int(s["units"][k][s["keep"]].sum()) for s in S) for k in ("rsUnits", "fsUnits")],
      sum(int((~s["units"]["rsUnits"][s["keep"]] & ~s["units"]["fsUnits"][s["keep"]]).sum()) for s in S))
sw = 1000 * np.concatenate([s["units"]["spike_width"][s["keep"]] for s in S])   # the values are in seconds
rs = np.concatenate([s["units"]["rsUnits"][s["keep"]] for s in S])
fs = np.concatenate([s["units"]["fsUnits"][s["keep"]] for s in S])
print(f"Spike width x 1000 (ms): RS {sw[rs].min():.3f} to {sw[rs].max():.3f}; FS {sw[fs].min():.3f} to {sw[fs].max():.3f}; "
      f"neither {sw[~rs & ~fs].min():.3f} to {sw[~rs & ~fs].max():.3f} (Methods: RS > 0.46, FS < 0.36)")

**Result: reproduced.** The unit counts are exactly the paper's: A1 780, wS1 1938, wS2 2064, wM2
2749 and ALM 2763, 10,294 in all; they match only with the CCF filter (without it, 12,066 units sit
on the five areas' probes). The probe session and mouse counts are the paper's (A1 14 and 6, wS1 24
and 12, wS2 24 and 11, wM2 18 and 8, ALM 16 and 6); for wS2 only 19 sessions in 9 mice keep units
after the CCF filter.

## fig3a-activity-maps

> In quiet Hit trials, probe-averaged activity changes are largest in A1 in the first 30 ms after the
> tone, in wM2 and ALM in the last 200 ms of the delay, in wS1 and wS2 in the first 30 ms after the
> whisker deflection, and in frontal areas 300 to 500 ms after it.

Per area-session (one probe each), the mean over neurons of the trial-averaged change in rate over
quiet Hit trials in the windows of the code (Ghaderi2025_Figure3_a_brainmap_activity.m, bins labelled
0.01-0.03, 0.8-1.0, 1.01-1.03 and 1.3-1.5 s); per area, the mean over probes. Pre-whisker baseline
readings (a) 0.95 to 1.0 s and (b) 0.8 to 1.0 s; the code's -1 to 0 s baseline for every window is a
variant.

In [ ]:
WIN3A = {"Auditory": (0.01, 0.03), "Delay": (0.8, 1.0), "Whisker": (1.01, 1.03), "Lick": (1.3, 1.5)}
TOP = {"Auditory": {"A1"}, "Delay": {"wM2", "ALM"}, "Whisker": {"wS1", "wS2"}, "Lick": {"wM2", "ALM"}}


def activity_maps(reading):
    rows = []
    for a in g.AREAS:
        for s, cols in g.area_sessions(S, a):
            sel = g.trial_mask(s, 1, True) & g.quiet_mask(s)
            if sel.sum() == 0:
                continue
            m = s["c10"][sel][:, :, cols].mean(axis=0) / g.W10
            row = {"area": a, "session": s["session"], "neurons": len(cols), "trials": int(sel.sum())}
            for k, (t1, t2) in WIN3A.items():
                if k in ("Auditory", "Delay") or reading == "code":
                    b = g.bin_index(-1.0, 0.0)
                else:
                    b = g.bin_index(0.95, 1.0) if reading == "a" else g.bin_index(0.8, 1.0)
                row[k] = (m[g.bin_index(t1, t2)].mean(axis=0) - m[b].mean(axis=0)).mean()
            rows.append(row)
    return pd.DataFrame(rows)


for reading in ("a", "b", "code"):
    df = activity_maps(reading)
    means = df.groupby("area")[list(WIN3A)].mean().loc[g.AREAS]
    print(f"Reading {reading}: mean over probes (Hz)")
    print(means.round(2).to_string())
    held = []
    for k, top in TOP.items():
        order = means[k].sort_values(ascending=False).index.tolist()
        ok = set(order[:len(top)]) == top
        held.append(ok)
        print(f"  {k}: largest {order[:len(top)]}, expected {sorted(top)} -> {ok}")
    print(f"  {sum(held)} of 4 hold\n")
print("Per probe, reading (a):")
print(activity_maps("a").round(2).to_string(index=False))

**Result: reproduced.** Under both pre-whisker readings, A1 has the largest change in the Auditory
window (5.55 Hz against at most 1.54), wM2 and ALM the two largest in the Delay window (0.70 and
0.35 Hz; A1 0.20), wS1 and wS2 the two largest in the Whisker window (6.80 and 5.61 Hz under (a))
and wM2 and ALM the two largest in the Lick window (1.79 and 1.80 Hz under (a); wS2 1.28). The code's
-1 to 0 s baseline gives the same ranking.

## fig3b-delay-and-response-psth

> In grand-average PSTHs of correct quiet trials, firing in wM2 and ALM is elevated during the delay of
> Go-tone trials compared with Nogo-tone trials, and after the whisker stimulus activity rises in
> Go-tone Whisker Hit trials compared with all other trial types in wS1, wS2, wM2 and ALM.

Per neuron, baseline-subtracted means over quiet correct trials; Wilcoxon signed-rank over neurons.
Pooled Go-tone trials are the Hit and Go-tone No-Whisker correct trials together, and likewise for
Nogo-tone. Neurons of sessions lacking either condition are left out (counts printed).

In [ ]:
hit = lambda s: g.correct_mask(s, 1)
go12 = lambda s: g.correct_mask(s, 1) | g.correct_mask(s, 2)
ng34 = lambda s: g.correct_mask(s, 3) | g.correct_mask(s, 4)
rows = []
for variant, sigs in (("main (whisker and jaw)", ("whisker_speed", "jaw_movement")), ("code (whisker only)", ("whisker_speed",))):
    for a in ("wM2", "ALM"):
        for pairing, (m1, m2) in {"pooled 1+2 vs 3+4": (go12, ng34), "type 1 vs 3": (hit, lambda s: g.correct_mask(s, 3))}.items():
            (x, y), sess = g.neuron_window_means(S, a, [m1, m2], (0.8, 1.0), signals=sigs)
            rows.append({"variant": variant, "part": "i delay", "area": a, "comparison": pairing, "window": "0.8-1.0",
                         "neurons": len(x), "sessions": len(set(sess)), "Go": x.mean(), "other": y.mean(),
                         "diff": (x - y).mean(), "p": stats.wilcoxon(x, y).pvalue})
    for window in ((1.3, 1.5), (1.0, 1.5)):
        for a in ("wS1", "wS2", "wM2", "ALM"):
            for t_ in (2, 3, 4, 5):
                (x, y), sess = g.neuron_window_means(S, a, [hit, lambda s, t_=t_: g.correct_mask(s, t_)], window, signals=sigs)
                rows.append({"variant": variant, "part": "ii response", "area": a, "comparison": f"Hit vs type {t_}",
                             "window": f"{window[0]}-{window[1]}", "neurons": len(x), "sessions": len(set(sess)),
                             "Go": x.mean(), "other": y.mean(), "diff": (x - y).mean(), "p": stats.wilcoxon(x, y).pvalue})
psth = pd.DataFrame(rows)
psth["holds"] = (psth["diff"] > 0) & (psth.p < 0.05)
print(psth.round(4).to_string(index=False))
for (variant, part, window), sub in psth.groupby(["variant", "part", "window"]):
    print(f"{variant}, {part}, {window}: {int(sub.holds.sum())} of {len(sub)} comparisons hold; failing: "
          f"{sub.loc[~sub.holds, ['area', 'comparison']].values.tolist()}")

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(15, 3), sharey=True)
t = labels10
ranges = []
for ax, a in zip(axs, g.AREAS):
    for t_, col in zip(range(1, 6), ("b", "c", "r", "orange", "grey")):
        traces = []
        for s, cols in g.area_sessions(S, a):
            m = g.correct_mask(s, t_) & g.quiet_mask(s)
            if m.sum() == 0:
                continue
            x = s["c10"][m][:, :, cols].mean(axis=0) / g.W10
            traces.append(x - x[g.bin_index(-1, 0)].mean(axis=0))
        mtrace = np.concatenate(traces, axis=1).mean(axis=1)
        ranges.append((mtrace.min(), mtrace.max()))
        ax.plot(t, mtrace, color=col, lw=0.8, label=g.TRIAL_NAMES[t_])
    ax.set_title(a)
    ax.set_xlim(-0.2, 1.6)
axs[0].set_ylabel("rate change (Hz)")
axs[0].legend(fontsize=6)
plt.show()
print(f"Plotted traces span {min(r[0] for r in ranges):.2f} to {max(r[1] for r in ranges):.2f} Hz over -1 to 2 s; x axis shows -0.2 to 1.6 s.")

**Result: partially reproduced.** (i) holds: in wM2 and ALM delay firing is higher in Go-tone than
Nogo-tone trials under both pairings (wM2 p 0.004 pooled and 0.0015 by type; ALM 0.009 and 0.008).
(ii) holds in wS1, wM2 and ALM, but in wS2 the Hit response over 1.3 to 1.5 s is not significantly
above No-tone Whisker trials (difference 0.80 Hz, p 0.11), so (ii) fails for one area. Over 1.0 to
1.5 s every comparison holds. The code's whisker-only quiet rule gives the same pattern.

## fig3c-delay-modulation-wm2-alm

> In the last 200 ms of the delay of quiet Hit trials, more wM2 than ALM neurons are up-modulated
> relative to baseline (16.0% against 14.2%, p = 0.0499) and fewer are down-modulated (14.3% against
> 17.0%, p = 0.0062), by chi-squared tests.

As code, Modulation.m: counts in 0.8 to 1.0 s against -0.2 to 0 s of the same trials (bins 181:200
and 81:100), SI = 2(AUC - 0.5), 200 label permutations. Three permutation seeds; the code's mean
absolute deviation as a variant.

In [ ]:
W_DELAY, W_PRE = np.arange(180, 200), np.arange(80, 100)


def modulation(area, rule, rng):
    si, p = [], []
    for s, cols in g.area_sessions(S, area):
        sel = g.trial_mask(s, 1, True) & g.quiet_mask(s, rule)
        if sel.sum() == 0:
            continue
        X = s["c10"][sel][:, :, cols].astype(float)
        w_, b_ = X[:, W_DELAY].sum(axis=1), X[:, W_PRE].sum(axis=1)
        for j in range(len(cols)):
            r = g.roc_test(w_[:, j], b_[:, j], 200, rng)
            si.append(r[0])
            p.append(r[1])
    return np.array(si), np.array(p)


rows = []
for rule in ("median", "mean"):
    for seed in (5, 6, 7):
        rng = np.random.default_rng(seed)
        R = {a: modulation(a, rule, rng) for a in ("wM2", "ALM")}
        for kind, sign in (("up", 1), ("down", -1)):
            k = {a: (np.sign(R[a][0]) == sign) & (R[a][1] < 0.05) for a in R}
            tab = [[k["wM2"].sum(), (~k["wM2"]).sum()], [k["ALM"].sum(), (~k["ALM"]).sum()]]
            rows.append({"MAD": rule, "seed": seed, "kind": kind, "wM2 %": 100 * k["wM2"].mean(), "ALM %": 100 * k["ALM"].mean(),
                         "wM2 n": len(k["wM2"]), "ALM n": len(k["ALM"]),
                         "chi2 p": stats.chi2_contingency(tab, correction=False).pvalue})
mod = pd.DataFrame(rows)
print(mod.round(4).to_string(index=False))
print("\nPaper: up 16.0 (wM2) and 14.2 (ALM), p 0.0499; down 14.3 and 17.0, p 0.0062. Tolerances 10%.")

**Result: partially reproduced.** Under the Methods' quiet rule both directions hold and both tests
are significant for every seed (up: wM2 13.6 to 13.8% against ALM 11.6 to 11.8%, p 0.020 to 0.037;
down: 10.9 to 11.1% against 13.6 to 14.0%, p 0.0006 to 0.0046), but all four fractions are below
tolerance. With the code's mean absolute deviation, which keeps more trials, the fractions are the
paper's (up 16.0 to 16.3% against 14.3%, down 14.0 to 14.3% against 17.0%), and the up-modulation
test gives p 0.047 to 0.077 depending on the seed, against the paper's 0.0499.

## fig3d-wm2-infragranular-delay

> During the late delay of Go-tone Whisker trials, wM2 infragranular neurons change their firing more
> than supragranular neurons (1.2 +/- 0.1 Hz against 0.03 +/- 0.10 Hz, p = 1.7 x 10^-7, two-sample
> t-test), while no layer difference is detected in A1, wS2 or ALM.

In [ ]:
LAYER_GROUP = {"1": "supra", "2/3": "supra", "4": "gran", "5": "infra", "6a": "infra", "6b": "infra"}
rows = []
for variant, rule, sigs in (("main", "median", ("whisker_speed", "jaw_movement")),
                            ("code quiet (whisker only)", "median", ("whisker_speed",)),
                            ("code MAD (mean)", "mean", ("whisker_speed", "jaw_movement"))):
    for a in g.AREAS:
        v, L = [], []
        for s, cols in g.area_sessions(S, a):
            sel = g.trial_mask(s, 1, True) & g.quiet_mask(s, rule, sigs)
            if sel.sum() == 0:
                continue
            x = s["c10"][sel][:, :, cols].mean(axis=0) / g.W10
            v.append(x[g.bin_index(0.8, 1.0)].mean(axis=0) - x[g.bin_index(-1, 0)].mean(axis=0))
            L.append(s["units"]["allenccf_area_layer"][s["keep"][cols]])
        v = np.concatenate(v)
        L = np.array([LAYER_GROUP.get(l, "none") for l in np.concatenate(L)])
        d = {k: v[L == k] for k in ("supra", "gran", "infra")}
        row = {"variant": variant, "area": a, "unassigned": int((L == "none").sum())}
        for k, x in d.items():
            row[f"{k} n"] = len(x)
            row[f"{k} mean"] = x.mean() if len(x) else np.nan
            row[f"{k} SEM"] = x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
        for p1, p2 in (("infra", "supra"), ("infra", "gran"), ("gran", "supra")):
            row[f"p {p1}-{p2}"] = stats.ttest_ind(d[p1], d[p2]).pvalue if len(d[p1]) > 1 and len(d[p2]) > 1 else np.nan
        rows.append(row)
layers = pd.DataFrame(rows)
print(layers.round(4).to_string(index=False))
for _, r in layers[layers.area == "wM2"].iterrows():
    print(f"{r['variant']}: wM2 infragranular against supragranular, t-test p = {r['p infra-supra']:.2e}")

**Result: partially reproduced.** In wM2 infragranular neurons change by 0.955 +/- 0.119 Hz and
supragranular by -0.055 +/- 0.093 Hz (t-test p 1.6e-06; wM2 has no layer 4 units); the
supragranular value is within tolerance, the infragranular one below it (1.1 to 1.3). No layer pair
differs in A1, wS2 or ALM (smallest p 0.21). With the code's mean absolute deviation the wM2 values
are 1.128 and -0.003 Hz (p 2.0e-07), within tolerance. The criterion's partial case does not cover
this result (the wM2 test holds and no other pair is significant, but one value is out of
tolerance); it is recorded as partial since the effect is there and one number misses.

## cluster-label-link

> The paper's cluster labels (Zenodo, processed_data/Data_Clustering.mat) can be matched to the
> dataset's units by the code's global unit index.

The global index is rebuilt as psth_10ms.m builds it (helper `global_ids`). The link holds when every
labelled neuron maps to a unit of the dataset whose probe area, CCF label and layer are the ones the
label file gives, and the unit is one of the 10,294 analysed here.

In [ ]:
t0 = time.time()
CL = g.cluster_labels()
print(f"read {len(CL['cluster'])} cluster labels in {time.time() - t0:.0f} s; clusters {CL['cluster'].min()} to {CL['cluster'].max()}")
GID = g.global_ids(S)
print("global indices built:", len(GID), "; units in the dataset:", sum(len(s["units"]["location"]) for s in S))
match = {"mapped": 0, "area": 0, "ccf": 0, "layer": 0, "analysed": 0}
unit_cluster = {}
for gi, cl, ar, cc, la in zip(CL["global_id"], CL["cluster"], CL["area"], CL["ccf"], CL["layer"]):
    if gi not in GID:
        continue
    i, u = GID[gi]
    U = S[i]["units"]
    match["mapped"] += 1
    match["area"] += U["location"][u] == ar
    match["ccf"] += U["ccf_location"][u] == cc
    match["layer"] += U["allenccf_area_layer"][u] == la
    match["analysed"] += U["area"][u] == ar
    unit_cluster[(i, u)] = cl
n = len(CL["cluster"])
print({k: f"{v} of {n}" for k, v in match.items()})
print("Distinct dataset units matched:", len(unit_cluster))

**Result: reproduced.** All 10,294 labelled neurons map to distinct dataset units with the same probe
area, CCF label and layer, and each is one of the 10,294 units analysed here.

## fig4b-named-clusters

> Some clusters represent sensory responses (cluster 1, fast auditory response, mostly A1 neurons;
> cluster 29, fast whisker response, mostly wS1 and wS2), clusters 11 and 25 fire during the delay only
> on Go-tone trials and consist mainly of wM2 and ALM neurons, and cluster 4 decreases its firing during
> the Go-tone delay and is spread over the five areas.

PSTHs as fig3b-delay-and-response-psth (quiet correct trials, baseline -1 to 0 s). The response of
clusters 1 and 29 is shown for each correct trial type that has the stimulus.

In [ ]:
def cluster_of(s_index, area):
    s = S[s_index]
    rows_ = s["keep"][g.area_columns(s, area)]
    return np.array([unit_cluster.get((s_index, int(u)), 0) for u in rows_])


comp = []
for a in g.AREAS:
    for i in range(len(S)):
        for c_ in cluster_of(i, a):
            comp.append((c_, a))
comp = pd.DataFrame(comp, columns=["cluster", "area"])
share = pd.crosstab(comp.cluster, comp.area, normalize="index")[g.AREAS] * 100
counts = pd.crosstab(comp.cluster, comp.area)[g.AREAS]
print("Neurons per cluster and area (%), clusters 1, 4, 11, 25, 29:")
print(pd.concat([counts.loc[[1, 4, 11, 25, 29]].sum(axis=1).rename("n"), share.loc[[1, 4, 11, 25, 29]].round(1)], axis=1).to_string())


def cluster_window(cl, masks, window):
    out = [[] for _ in masks]
    for a in g.AREAS:
        for i, s in enumerate(S):
            cols = g.area_columns(s, a)
            if len(cols) == 0:
                continue
            sel_cols = cols[cluster_of(i, a) == cl]
            if len(sel_cols) == 0:
                continue
            q = g.quiet_mask(s)
            for k, m in enumerate(masks):
                mm = m(s) & q
                if mm.sum() == 0:
                    continue
                x = s["c10"][mm][:, :, sel_cols].mean(axis=0) / g.W10
                out[k].append(x[g.bin_index(*window)].mean(axis=0) - x[g.bin_index(-1, 0)].mean(axis=0))
    return [np.concatenate(o) for o in out]


tone_types = [lambda s, t_=t_: g.correct_mask(s, t_) for t_ in (1, 2, 3, 4)]
whisk_types = [lambda s, t_=t_: g.correct_mask(s, t_) for t_ in (1, 3, 5)]
r1 = cluster_window(1, tone_types, (0.01, 0.03))
r29 = cluster_window(29, whisk_types, (1.01, 1.03))
print("\nCluster 1, mean change 0 to 30 ms (Hz), trial types 1-4:", [round(x.mean(), 2) for x in r1])
print("Cluster 29, mean change 1.00 to 1.03 s (Hz), trial types 1, 3, 5:", [round(x.mean(), 2) for x in r29])
for cl in (11, 25, 4):
    h_, n_ = cluster_window(cl, [lambda s: g.correct_mask(s, 1), lambda s: g.correct_mask(s, 3)], (0.8, 1.0))
    print(f"Cluster {cl}, 0.8 to 1.0 s: Hit {h_.mean():.2f} Hz, Nogo-tone Whisker CR {n_.mean():.2f} Hz "
          f"(neurons {len(h_)}, {len(n_)})")


def mostly(cl, group):
    sh = share.loc[cl]
    a_ = sh[list(group)].sum() > 50
    b_ = set(sh.sort_values(ascending=False).index[:len(group)]) == set(group)
    return a_, b_


for cl, group in ((1, ("A1",)), (29, ("wS1", "wS2")), (11, ("wM2", "ALM")), (25, ("wM2", "ALM"))):
    a_, b_ = mostly(cl, group)
    print(f"Cluster {cl} mostly {group}: reading (a) more than half: {a_}; reading (b) largest share: {b_}")
print("Cluster 4, share in each area (%):", share.loc[4].round(1).to_dict(), "; at least 5% in each:", bool((share.loc[4] >= 5).all()))

**Result: blocked (methods underspecified).** Parts (2) to (4) hold under both readings of "mostly":
cluster 29 is 45.2% wS1 and 41.5% wS2 with a whisker response of 21.7 to 21.9 Hz; clusters 11 and 25
are 90.4% and 83.1% wM2 and ALM and fire more in Hit than in Nogo CR trials over the late delay
(11.54 against -0.79 Hz, 3.56 against -0.29 Hz); cluster 4 fires less in Hit (-1.60 against 0.05 Hz)
and has 7.1 to 31.9% of its neurons in each area. Cluster 1 has a fast auditory response (13.0 to
17.3 Hz) and its largest share in A1 (30.5%), but not more than half: reading (b) gives reproduced
and reading (a) partially reproduced. The criterion blocks the claim when the readings differ.

## fig5a-single-neuron-context

> Comparing Hit with Nogo-tone Whisker correct rejection trials in the last 200 ms of the delay, wM2
> and ALM have the highest proportions of context-modulated neurons (wM2, 20% increase firing by 8 Hz
> on average, 14% decrease by 3 Hz; ALM, 17% increase by 8 Hz, 16% decrease by 3 Hz), and wM2 has the
> largest proportion of Go-tone excited neurons.

As code, Roc_hit_cr3.m: counts in bins 181:200 (0.8 to 1.0 s) of quiet Hit against quiet Nogo-tone
Whisker CR trials, SI and 200-permutation p; the change is the Hit minus CR mean rate.

In [ ]:
rows = []
for rule in ("median", "mean"):
    rng = np.random.default_rng(8)
    for a in g.AREAS:
        si, p, d = [], [], []
        for s, cols in g.area_sessions(S, a):
            q = g.quiet_mask(s, rule)
            m1, m2 = g.trial_mask(s, 1, True) & q, g.trial_mask(s, 3, False) & q
            if m1.sum() == 0 or m2.sum() == 0:
                continue
            x = s["c10"][m1][:, W_DELAY][:, :, cols].sum(axis=1).astype(float)
            y = s["c10"][m2][:, W_DELAY][:, :, cols].sum(axis=1).astype(float)
            for j in range(len(cols)):
                r = g.roc_test(x[:, j], y[:, j], 200, rng)
                si.append(r[0])
                p.append(r[1])
            d.append(x.mean(axis=0) / 0.2 - y.mean(axis=0) / 0.2)
        si, p, d = np.array(si), np.array(p), np.concatenate(d)
        up, dn = (si > 0) & (p < 0.05), (si < 0) & (p < 0.05)
        rows.append({"MAD": rule, "area": a, "neurons": len(si), "up %": 100 * up.mean(), "up mean Hz": d[up].mean(),
                     "down %": 100 * dn.mean(), "down mean Hz": d[dn].mean(), "modulated %": 100 * (up | dn).mean()})
ctx = pd.DataFrame(rows)
print(ctx.round(2).to_string(index=False))
exp5a = {("wM2", "up %"): (20, 2), ("wM2", "up mean Hz"): (8, 0.8), ("wM2", "down %"): (14, 1.4), ("wM2", "down mean Hz"): (-3, 0.5),
         ("ALM", "up %"): (17, 1.7), ("ALM", "up mean Hz"): (8, 0.8), ("ALM", "down %"): (16, 1.6), ("ALM", "down mean Hz"): (-3, 0.5)}
for rule in ("median", "mean"):
    sub = ctx[ctx.MAD == rule].set_index("area")
    within = {f"{a} {k}": bool(abs(sub.loc[a, k] - v) <= tol) for (a, k), (v, tol) in exp5a.items()}
    print(f"\n{rule}: within tolerance {sum(within.values())} of 8: {within}")
    print(f"  wM2 up fraction largest: {sub['up %'].idxmax() == 'wM2'}; wM2 and ALM modulated above A1, wS1, wS2: "
          f"{min(sub.loc['wM2', 'modulated %'], sub.loc['ALM', 'modulated %']) > sub.loc[['A1', 'wS1', 'wS2'], 'modulated %'].max()}")

**Result: partially reproduced.** Under the Methods' quiet rule the wM2 up-modulated fraction is
the largest (17.9%) and wM2 (29.4%) and ALM (27.4%) have more modulated neurons than A1, wS1 and wS2
(at most 15.5%), and the four firing changes are within tolerance (8.65, -3.16, 8.05, -3.40 Hz), but
the four fractions are below it (wM2 17.9% and 11.5%, ALM 14.5% and 12.9%). With the code's mean
absolute deviation all eight values are within tolerance (wM2 20.0%, 8.14 Hz, 14.1%, -2.77 Hz; ALM
16.7%, 7.53 Hz, 16.1%, -2.97 Hz).

## fig6a-pca-delay-states

> In the first two PCs, A1, wS1 and wS2 activity returns near its start by the end of the delay, while in
> correct Go-tone trials wM2 and ALM settle within about 200 ms into a state far from the origin,
> reached with and without the whisker stimulus; Nogo correct rejections return to baseline and Miss
> trials fail to reach the Go-tone state. PC1 explains 11% (A1, wS1, wS2), 17% (wM2) and 14% (ALM) of
> variance, PC2 6% (A1, wS1, wS2) and 8% (wM2, ALM).

As code, Ghaderi2025_Figure6_a_pca.m: per area all neurons, mean over quiet completed trials of Hit,
Nogo-tone Whisker CR, Go-tone No-Whisker CR and Miss in 50 ms bins, baseline (-1 to 0 s) subtracted,
the four conditions concatenated (240 time points), neurons with no range (or no trials of a
condition) dropped, each divided by its range; PCA over time points.

In [ ]:
CONDS6 = [("Hit", 1, True), ("Nogo CR", 3, False), ("Go NoW CR", 2, False), ("Miss", 1, False)]


def pca_area(area, rule, fit_on=None):
    blocks = []
    for s, cols in g.area_sessions(S, area):
        q = g.quiet_mask(s, rule)
        M = []
        for _, t_, l_ in CONDS6:
            m = g.trial_mask(s, t_, l_) & q
            if m.sum():
                x = s["c10"][m][:, :, cols].astype(float).reshape(m.sum(), 60, 5, len(cols)).sum(axis=2).mean(axis=0)
            else:
                x = np.full((60, len(cols)), np.nan)
            M.append(x - x[:20].mean(axis=0))
        blocks.append(np.concatenate(M, axis=0))
    X = np.concatenate(blocks, axis=1).T
    rng_ = X.max(axis=1) - X.min(axis=1)
    ok = np.isfinite(rng_) & (rng_ > 0)
    X = (X[ok] / rng_[ok, None]).T
    fit = X if fit_on is None else X[np.concatenate([np.arange(i * 60, (i + 1) * 60) for i in fit_on])]
    mu = fit.mean(axis=0)
    _, sv, Vt = np.linalg.svd(fit - mu, full_matrices=False)
    ev = 100 * sv ** 2 / (sv ** 2).sum()
    P = (X - X.mean(axis=0)) @ Vt[:2].T if fit_on is None else (X - mu) @ Vt[:2].T
    dist = []
    for i in range(4):
        seg = P[i * 60:(i + 1) * 60]
        dist.append(np.linalg.norm(seg[36:40].mean(axis=0) - seg[:20].mean(axis=0)))
    hitseg = P[:60]
    dh = np.linalg.norm(hitseg - hitseg[:20].mean(axis=0), axis=1)
    return {"area": area, "neurons kept": int(ok.sum()), "neurons": len(ok), "PC1 %": ev[0], "PC2 %": ev[1],
            **{f"dist {c[0]}": d_ for c, d_ in zip(CONDS6, dist)}, "Hit dist at 1.0 / max 0-1 s": dh[39] / dh[20:40].max()}, P


exp6 = {"A1": (11, 6), "wS1": (11, 6), "wS2": (11, 6), "wM2": (17, 8), "ALM": (14, 8)}
pca_rows, projections = [], {}
for variant, rule, fit_on in (("main", "median", None), ("code MAD (mean)", "mean", None), ("legend: fit on Hit and Nogo CR", "median", (0, 1))):
    for a in g.AREAS:
        r, P_ = pca_area(a, rule, fit_on)
        r["variant"] = variant
        r["PC1 within"] = abs(r["PC1 %"] - exp6[a][0]) <= 0.1 * exp6[a][0]
        r["PC2 within"] = abs(r["PC2 %"] - exp6[a][1]) <= 0.1 * exp6[a][1]
        if a in ("wM2", "ALM"):
            r["geometry"] = min(r["dist Hit"], r["dist Go NoW CR"]) > max(r["dist Nogo CR"], r["dist Miss"])
        else:
            r["geometry"] = r["Hit dist at 1.0 / max 0-1 s"] < 0.5
        pca_rows.append(r)
        if variant == "main":
            projections[a] = P_
pca_tab = pd.DataFrame(pca_rows)
print(pca_tab.round(3).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(15, 3))
lo, hi = [], []
for ax, a in zip(axs, g.AREAS):
    P_ = projections[a]
    for i, (name, *_), col in zip(range(4), CONDS6, ("b", "r", "c", "purple")):
        seg = P_[i * 60:(i + 1) * 60][20:40]
        ax.plot(seg[:, 0], seg[:, 1], color=col, label=name)
        ax.plot(seg[0, 0], seg[0, 1], "o", color=col, ms=3)
    lo.append(P_.min())
    hi.append(P_.max())
    ax.set_title(a)
axs[0].legend(fontsize=6)
plt.show()
print(f"Delay trajectories (0 to 1 s) in PC1-PC2; projections span {min(lo):.2f} to {max(hi):.2f}.")

**Result: partially reproduced.** The geometry holds: in wM2 and ALM the end-of-delay state of Hit
and Go-tone No-Whisker trials is far from baseline (distances 7.14 and 5.87 in wM2, 6.64 and 5.65 in
ALM) while Nogo CR (0.96, 0.45) and Miss (1.90, 1.95) stay near it, and in A1, wS1 and wS2 the Hit
trajectory is back to 0.05 to 0.16 of its maximum distance at 1.0 s. Seven of the ten explained
variances are within tolerance under the Methods' quiet rule (PC1 wS1 9.8 and wM2 15.0% and PC2 ALM
6.8% are below it; PC1 ALM 12.61% is just inside). With the code's mean absolute deviation all ten
are within tolerance (PC1 11.1, 10.6, 11.5, 16.6, 14.3%; PC2 6.0, 6.1, 6.2, 8.2, 7.8%). The legend's
reading (PCs fitted on Hit and Nogo CR only) gives larger explained variances (PC1 16.7 to 27.2%).

## fig6b-context-coding-direction

> Projected on a context coding direction, late-delay activity in wM2 and ALM is high in Hit trials and
> significantly lower in Nogo-tone, Go-tone No-Whisker and Miss trials.

As the criterion: per area-session with at least 5 neurons, CD context from a random 20% of quiet Hit
and of quiet Nogo CR trials (Methods; 30% in the code, a variant), the mean over bins labelled 0.8
to 1.0 s of the difference of mean count vectors, unit length (Gram-Schmidt with the later
directions leaves it unchanged); held-out Hit and Nogo CR trials and all Go-tone No-Whisker CR and
Miss trials projected; per session the mean projection over 0.8 to 1.0 s; Wilcoxon, p times 3.

In [ ]:
WCTX = g.bin_index(0.8, 1.0)


def cd_context(frac, rule, rng):
    rows = []
    for a in g.AREAS:
        for s, cols in g.area_sessions(S, a, 5):
            q = g.quiet_mask(s, rule)
            X = s["c10"][:, :, cols].astype(float)
            ids = {name: np.flatnonzero(g.trial_mask(s, t_, l_) & q) for name, t_, l_ in CONDS6}
            h_tr, h_te = g.split_train(ids["Hit"], frac, rng)
            c_tr, c_te = g.split_train(ids["Nogo CR"], frac, rng)
            if len(h_tr) == 0 or len(c_tr) == 0:
                rows.append({"area": a, "session": s["session"], "dropped": True})
                continue
            cd = X[h_tr][:, WCTX].mean(axis=(0, 1)) - X[c_tr][:, WCTX].mean(axis=(0, 1))
            cd /= np.linalg.norm(cd)
            pr = lambda t: (X[t][:, WCTX] @ cd).mean() if len(t) else np.nan
            rows.append({"area": a, "session": s["session"], "dropped": False, "Hit": pr(h_te), "Nogo CR": pr(c_te),
                         "Go NoW CR": pr(ids["Go NoW CR"]), "Miss": pr(ids["Miss"])})
    return pd.DataFrame(rows)


def hit_vs(df, others, factor=3):
    out = []
    for a in g.AREAS:
        sub = df[(df.area == a) & ~df.dropped]
        for o in others:
            ok = sub["Hit"].notna() & sub[o].notna()
            p_ = stats.wilcoxon(sub.loc[ok, "Hit"], sub.loc[ok, o]).pvalue
            out.append({"area": a, "vs": o, "sessions": int(ok.sum()), "Hit": sub.loc[ok, "Hit"].mean(),
                        "other": sub.loc[ok, o].mean(), "diff": (sub.loc[ok, "Hit"] - sub.loc[ok, o]).mean(),
                        "p x3": min(1.0, factor * p_)})
    return pd.DataFrame(out)


for frac, rule, seed in ((0.2, "median", 9), (0.3, "median", 9), (0.2, "mean", 9), (0.2, "median", 10)):
    df = cd_context(frac, rule, np.random.default_rng(seed))
    res = hit_vs(df, ["Nogo CR", "Go NoW CR", "Miss"])
    res["holds"] = (res["diff"] > 0) & (res["p x3"] < 0.05)
    print(f"Training fraction {frac}, MAD {rule}, seed {seed}; area-sessions dropped (no training trials): {int(df.dropped.sum())}")
    print(res.round(4).to_string(index=False))
    print("  wM2 and ALM, comparisons holding:", int(res[res.area.isin(["wM2", "ALM"])].holds.sum()), "of 6\n")

**Result: reproduced.** With 20% training trials and the Methods' quiet rule, Hit projections in wM2
exceed Nogo CR, Go-tone No-Whisker CR and Miss (corrected p below 0.0001, 0.0010 and below 0.0001)
and in ALM likewise (0.0001, 0.0039, 0.0002); all six comparisons hold under the variants (30%
training, the code's MAD, another seed).

## fig6c-temporal-correlation

> In wM2 and ALM the Go-tone induces a long-lasting increase in correlated population activity during
> the delay in Hit and Go-tone No-Whisker trials, absent in Nogo-tone and Miss trials, while in A1, wS1
> and wS2 the cue evokes only a brief increase.

As the criterion (code, Temporal_corrrelation_10ms.m and Ghaderi2025_Figure6_c_temporalCorrelation.m).

In [ ]:
off = np.ones((100, 100)) - np.eye(100)
off[off == 0] = np.nan
rows, maps = [], {}
for a in g.AREAS:
    for s, cols in g.area_sessions(S, a, 5):
        q = g.quiet_mask(s)
        row = {"area": a, "session": s["session"]}
        for name, t_, l_ in CONDS6:
            m = g.trial_mask(s, t_, l_) & q
            if m.sum() == 0:
                row[name] = np.nan
                continue
            R = g.temporal_correlation(s["c10"][m][:, :, cols])
            R = R - np.nanmean(R[:100, :100] * off)
            row[name] = np.nanmean(R[99:200, 99:200])
            maps.setdefault((a, name), []).append(R)
        rows.append(row)
tc = pd.DataFrame(rows).assign(dropped=False)
res = hit_vs(tc, ["Nogo CR", "Go NoW CR", "Miss"])
res["holds"] = (res["diff"] > 0) & (res["p x3"] < 0.05)
print(res.round(5).to_string(index=False))
a1 = tc[tc.area == "A1"].reset_index(drop=True)
print("\nA1 without its fourth Hit session (as the code does):")
print(hit_vs(a1.drop(index=3).assign(area="A1"), ["Nogo CR", "Go NoW CR", "Miss"]).query("area == 'A1'").round(4).to_string(index=False))

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(15, 6))
vals = []
for j, a in enumerate(g.AREAS):
    for i, name in enumerate(("Hit", "Nogo CR")):
        M = np.nanmean(maps[(a, name)], axis=0)
        vals.append(M)
        axs[i, j].imshow(M, origin="lower", extent=[-1, 2, -1, 2], vmin=-0.02, vmax=0.08, cmap="viridis")
        axs[i, j].set_title(f"{a} {name}", fontsize=8)
allv = np.concatenate([v.ravel() for v in vals])
print(f"Mean maps span {np.nanmin(allv):.3f} to {np.nanmax(allv):.3f}; colour scale -0.02 to 0.08, "
      f"{int(np.sum(allv > 0.08) + np.sum(allv < -0.02))} of {np.isfinite(allv).sum()} values outside it (mostly the diagonal).")
plt.show()

**Result: reproduced.** In wM2 the baseline-subtracted delay correlation of Hit trials exceeds Nogo
CR, Go-tone No-Whisker CR and Miss (corrected p 2.3e-05, 0.0039, 4.6e-05; 18, 18 and 17 sessions)
and in ALM likewise (0.00018, 0.00027, 0.00037). The brief cue-evoked increase in A1, wS1 and wS2 is
visible in the maps and is not judged.

## fig7b-early-whisker-response

> Whisker-evoked activity in the Go-tone context is transiently reduced in wS1 and wS2 around 15 ms after
> the deflection and higher from about 30 ms on in most areas, compared with the Nogo-tone context.

As the criterion (code, psth_5ms.m and Ghaderi2025_Figure7_b_DeltaFR.m). Bins are labelled by their
right edge, as in the code.

In [ ]:
lab5 = g.T5_START + g.W5 * np.arange(1, g.N5 + 1)
rows, dfr = [], {}
for a in g.AREAS:
    D = []
    for s, cols in g.area_sessions(S, a):
        q = g.quiet_mask(s)
        m1, m2 = g.trial_mask(s, 1, True) & q, g.trial_mask(s, 3, False) & q
        if m1.sum() == 0 or m2.sum() == 0:
            continue
        x = s["c5"][m1][:, :, cols].mean(axis=0) / g.W5
        y = s["c5"][m2][:, :, cols].mean(axis=0) / g.W5
        D.append((x - x[:10].mean(axis=0)) - (y - y[:10].mean(axis=0)))
    D = np.concatenate(D, axis=1)
    dfr[a] = D
    for i in range(g.N5):
        rows.append({"area": a, "bin": round(lab5[i], 3), "neurons": D.shape[1], "mean dFR": D[i].mean(),
                     "p": stats.wilcoxon(D[i]).pvalue if np.any(D[i] != 0) else 1.0})
b7 = pd.DataFrame(rows)
b7["p_FDR_area"] = b7.groupby("area").p.transform(lambda p: g.bh_fdr(p.values))
b7["p_FDR_all"] = g.bh_fdr(b7.p.values)
for col in ("p_FDR_area", "p_FDR_all"):
    print(f"FDR {col}:")
    for a in g.AREAS:
        sub = b7[b7.area == a]
        neg = sub[(sub[col] < 0.05) & (sub["mean dFR"] < 0)].bin.tolist()
        pos = sub[(sub[col] < 0.05) & (sub["mean dFR"] > 0)].bin.tolist()
        print(f"  {a}: significant negative bins {neg}; positive {pos}")
early_neg = {a: bool(((b7.area == a) & (b7.p_FDR_area < 0.05) & (b7["mean dFR"] < 0) & b7.bin.between(1.0099, 1.0201)).any()) for a in ("wS1", "wS2")}
late_pos = [a for a in g.AREAS if ((b7.area == a) & (b7.p_FDR_area < 0.05) & (b7["mean dFR"] > 0) & (b7.bin >= 1.0299)).any()]
print("\nwS1 and wS2 negative bin within 1.010-1.020 s:", early_neg)
print("Areas with significant positive bins from 1.030 s:", late_pos, f"({len(late_pos)} of 5)")
print(b7[b7.bin.between(1.0, 1.04)].pivot(index="bin", columns="area", values="mean dFR").round(2).to_string())

**Result: reproduced.** With FDR within each area, wS1 has significant negative Hit minus CR bins at
1.015 and 1.020 s and wS2 at 1.010 to 1.025 s, and all five areas have significant positive bins
from 1.030 s on (ALM from 1.030, wM2 from 1.035, wS1 and wS2 from 1.040, A1 from 1.065 s). The
code's FDR over all areas and bins gives the same bins for wS1, wS2, wM2 and ALM.

## fig7c-no-whisker-cd-change

> Activity projected on a whisker coding direction, defined from the first 30 ms of No-tone Whisker
> responses, shows no significant difference between Go-tone and Nogo-tone contexts.

As the criterion (code, Coding_direction_5ms.m and Ghaderi2025_Figure7_c_StimCD.m), with 30% training
trials (code). The 5 ms counts kept by the helper start at 0.95 s, so the -1 to 0 s baseline of the
whisker direction and the 0.8 to 1.0 s window of the context direction are taken from the 10 ms counts
divided by 2 (the mean count per 5 ms); the context window is then [0.79, 1.0) in place of
[0.795, 1.0). The projection is the 5 ms bin labelled 1.030 s.

In [ ]:
B10 = g.bin_index(-1.0, 0.0)
I_WHISK = np.arange(9, 16)      # 5 ms bins labelled 1.000 to 1.030 s
I_30 = 15                       # bin labelled 1.030 s
GROUPS8 = [("Go lick", 1, True), ("Go no lick", 1, False), ("Nogo lick", 3, True), ("Nogo no lick", 3, False),
           ("No-tone lick", 5, True), ("No-tone no lick", 5, False)]


def coding_directions_5ms(frac, rule, rng):
    rows7, rows8 = [], []
    for a in g.AREAS:
        for s, cols in g.area_sessions(S, a, 5):
            q = g.quiet_mask(s, rule)
            X10 = s["c10"][:, :, cols].astype(float)
            X5 = s["c5"][:, :, cols].astype(float)
            hit = np.flatnonzero(g.trial_mask(s, 1, True) & q)
            cr = np.flatnonzero(g.trial_mask(s, 3, False) & q)
            wt = np.flatnonzero(g.trial_mask(s, 5, False) & q)
            h_tr, h_te = g.split_train(hit, frac, rng)
            c_tr, c_te = g.split_train(cr, frac, rng)
            w_tr, w_te = g.split_train(wt, frac, rng)
            if min(len(h_tr), len(c_tr), len(w_tr)) == 0:
                continue
            cdc = (X10[h_tr][:, WCTX].mean(axis=(0, 1)) - X10[c_tr][:, WCTX].mean(axis=(0, 1))) / 2
            cdc /= np.linalg.norm(cdc)
            cdw = X5[w_tr][:, I_WHISK].mean(axis=(0, 1)) - X10[w_tr][:, B10].mean(axis=(0, 1)) / 2
            cdw /= np.linalg.norm(cdw)
            cdw = cdw - (cdw @ cdc) * cdc
            cdw /= np.linalg.norm(cdw)
            pr = lambda t, cd: (X5[t][:, I_30] @ cd).mean() if len(t) else np.nan
            rows7.append({"area": a, "session": s["session"], "Hit": pr(h_te, cdw), "Nogo CR": pr(c_te, cdw)})
            train = set(h_tr) | set(c_tr)
            row = {"area": a, "session": s["session"]}
            for name, t_, l_ in GROUPS8:
                ids = np.array([i for i in np.flatnonzero(g.trial_mask(s, t_, l_) & q) if i not in train], int)
                row[name] = pr(ids, cdc)
            rows8.append(row)
    return pd.DataFrame(rows7), pd.DataFrame(rows8)


cd7, cd8 = {}, {}
for key, (frac, rule, seed) in {"main": (0.3, "median", 11), "20% training": (0.2, "median", 11),
                                "code MAD (mean)": (0.3, "mean", 11), "seed 12": (0.3, "median", 12)}.items():
    cd7[key], cd8[key] = coding_directions_5ms(frac, rule, np.random.default_rng(seed))
for key, df in cd7.items():
    rows = []
    for a in g.AREAS:
        sub = df[df.area == a]
        H, p_, _ = g.kruskal_lsd([sub["Hit"], sub["Nogo CR"]])
        ok = sub["Hit"].notna() & sub["Nogo CR"].notna()
        rows.append({"area": a, "sessions": len(sub), "Hit": sub["Hit"].mean(), "Nogo CR": sub["Nogo CR"].mean(),
                     "diff": (sub["Hit"] - sub["Nogo CR"]).mean(), "KW p": p_,
                     "paired Wilcoxon p": stats.wilcoxon(sub.loc[ok, "Hit"], sub.loc[ok, "Nogo CR"]).pvalue})
    print(f"Whisker CD at 30 ms, {key}:")
    print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** The Kruskal-Wallis test finds no difference between Hit and Nogo CR
projections on the whisker direction in any area (p 0.31 to 0.97), as the claim of absence
requires, and the variants agree (smallest p 0.19). The paired Wilcoxon test, not the paper's,
gives p 0.096 in wS2 (Hit higher by 0.044) and p above 0.15 elsewhere; in the variants it falls
below 0.05 in wS2 (20% training, code's MAD) and A1 (code's MAD).

## fig8a-wm2-lick-divergence

> In the first 30 ms after the whisker stimulus, only wM2 activity diverges into separate states for lick
> and no-lick trials, mainly along the context coding direction, and at 30 ms distinguishes lick from
> no-lick trials both in correct Go-tone and error Nogo-tone conditions; ALM shows a weaker effect and A1,
> wS1 and wS2 none.

Held-out quiet trials projected on CD context (computed above, 30% training as the 5 ms code), the
value in the bin labelled 1.030 s, six groups; Kruskal-Wallis then Fisher LSD on the ranks.

In [ ]:
for key, df in cd8.items():
    rows = []
    for a in g.AREAS:
        sub = df[df.area == a]
        H, p_, comps = g.kruskal_lsd([sub[name] for name, *_ in GROUPS8])
        rows.append({"area": a, "sessions": len(sub), **{name: sub[name].mean() for name, *_ in GROUPS8},
                     "KW p": p_, "Go lick-nolick p": comps[(0, 1)][1], "Nogo lick-nolick p": comps[(2, 3)][1],
                     "No-tone lick-nolick p": comps[(4, 5)][1]})
    res8 = pd.DataFrame(rows)
    print(f"Context CD at 30 ms, {key}:")
    print(res8.round(4).to_string(index=False))
    print("  area-sessions (all areas) with trials in each group:", {name: int(df[name].notna().sum()) for name, *_ in GROUPS8}, "\n")

**Result: partially reproduced.** In wM2 the six groups differ (Kruskal-Wallis p 0.0048) and
Go-tone lick trials project above Go-tone no-lick trials (LSD p 0.0041), but Nogo-tone lick (false
alarm) trials are not significantly above Nogo-tone no-lick trials (p 0.28): wM2 holds for one of
the two contexts. A1, wS1 and wS2 show no significant lick against no-lick comparison (all p 0.40 or
more). ALM gives Kruskal-Wallis p below 0.0001, Go-tone p 0.0008 and Nogo-tone p 0.028. The
variants agree that the Nogo-tone comparison in wM2 is not significant (p 0.08 to 0.36).

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig1c-lick-probabilities | reproduced |
| fig1f-movement-latencies | partially reproduced |
| quiet-trials-half | partially reproduced |
| fig2-auditory-window | reproduced |
| fig2-delay-window | reproduced |
| fig2-whisker-window | reproduced |
| fig2-wm2-vs-alm-auditory | reproduced |
| fig2-other-trial-types | not reproduced |
| units-recorded | reproduced |
| fig3a-activity-maps | reproduced |
| fig3b-delay-and-response-psth | partially reproduced |
| fig3c-delay-modulation-wm2-alm | partially reproduced |
| fig3d-wm2-infragranular-delay | partially reproduced |
| cluster-label-link | reproduced |
| fig4b-named-clusters | blocked (methods underspecified) |
| fig5a-single-neuron-context | partially reproduced |
| fig6a-pca-delay-states | partially reproduced |
| fig6b-context-coding-direction | reproduced |
| fig6c-temporal-correlation | reproduced |
| fig7b-early-whisker-response | reproduced |
| fig7c-no-whisker-cd-change | reproduced |
| fig8a-wm2-lick-divergence | partially reproduced |
| fig8b-lick-coding-direction, figs18-later-lick-coding | blocked |

**How the analysis differs from the paper's.** Quiet trials follow the Methods (median absolute
deviation); the paper's code uses MATLAB's default mean absolute deviation, and with it the
single-neuron fractions (Fig. 3c, 5a), the layer values (Fig. 3d) and the explained variances (Fig.
6a) come out at the paper's values, while with the Methods' rule they are lower. The 5 ms
coding-direction analyses take their baseline and context window from the 10 ms counts. Permutation
and bootstrap seeds are fixed per section.

**What was found about the dataset.** The units, trials, optogenetic sessions and unit counts per
area are those of the paper, and the paper's cluster labels link one to one to the units. The
genotype field of 12 optogenetic sessions (PG021, PG061) says C57BL/6JRj; one session has 11 trials
of type 0; two sessions have trials without video; several trial-table descriptions do not fit
their columns; the area of a unit needs the CCF label list from the Zenodo record, which the
dandiset does not hold.